In [1]:
from pathlib import Path
import numpy as np
import nibabel as nib
import pyvista as pv
from nibabel.nifti1 import Nifti1Image

# import logging
# 强制重新配置，设置级别为 DEBUG
# logging.basicConfig(level=logging.DEBUG, force=True)


root_dir = Path.cwd().parent

cavity_path = root_dir / "test" / "test_data" / "volume_with_dvf" / "cavity.nii.gz"
coronary_path = root_dir / "test" / "test_data" / "volume_with_dvf" / "coronary.nii.gz"
image_path = root_dir / "test" / "test_data" / "volume_with_dvf" / "image.nii.gz"

ssm_dir = root_dir / "generate_4d_heart" /"ssm" / "ssm_world"
ssm_template_path = ssm_dir / "ssm_template.vtk"
b_motion_path = ssm_dir / "b_motion.npy"
P_motion_path = ssm_dir / "P_motion.npy"

(temp_output := Path("temp")).mkdir(parents=True, exist_ok=True)


def read_nii(path: Path) -> tuple[Nifti1Image, np.ndarray]:
    img = nib.load(path)
    assert isinstance(img, Nifti1Image)
    return img, img.get_fdata()

cavity_img, cavity_data = read_nii(cavity_path)
coronary_img, coronary_data = read_nii(coronary_path)
image_img, image_data = read_nii(image_path)

ssm_template: pv.PolyData = pv.read(ssm_template_path)  #type: ignore
assert isinstance(ssm_template, pv.PolyData)
b_motion: np.ndarray = np.load(b_motion_path)
P_motion: np.ndarray = np.load(P_motion_path)

In [ ]:
from generate_4d_heart import CavityLabel
import cupy as cp
from cupyx.scipy import ndimage
import pyvista as pv

# 使用形态学膨胀的方法无法得到准确的心腔区域
# def extract_pericardium(cavity_data: np.ndarray) -> np.ndarray:
#     # 得到心腔总标签
#     cavity_mask = np.isin(cavity_data, list(CavityLabel)).astype(np.uint8)
    
#     # 多次闭运算
#     structure = cp.ones((5,5,5), dtype=bool)  # 3D结构元素
#     cavity_mask_cp = cp.asarray(cavity_mask)
#     cavity_mask_cp = ndimage.binary_closing(cavity_mask_cp, structure=structure, iterations=10, brute_force=True)
#     return cp.asnumpy(cavity_mask_cp).astype(np.uint8)


# 尝试使用外包络的方法得到心腔区域
def generate_pericardium(cavity_data: np.ndarray, coronary_data: np.ndarray, padding_voxel: float=5.0) -> tuple[pv.PolyData, np.ndarray]:
    """
    使用 PyVista 提取并平滑心包包络
    """
    cavity_all_mask = np.isin(cavity_data, list(CavityLabel)).astype(np.uint8)
    cavity_add_coronary_mask = np.logical_or(cavity_all_mask, coronary_data > 0).astype(np.uint8)

    # 2. 提取等值面 (Marching Cubes)
    cavity_all_mesh: pv.PolyData = pv.wrap(cavity_add_coronary_mask)\
        .contour([1], method="flying_edges")\
        .triangulate()\
        .smooth_taubin()\
        .decimate_pro(
            reduction=0.95,          # 减少 三角面片
            preserve_topology=True, # 防止破洞
        )\
        .clean()

    points = cavity_all_mesh.points
    normals = cavity_all_mesh.point_normals
    offset_points = points + normals * padding_voxel

    pericardium_mesh = pv.PolyData(offset_points)\
        .delaunay_3d()\
        .extract_surface(algorithm=None)
    
    ref_volume = pv.ImageData(dimensions=cavity_all_mask.shape)
    pericardium_mask = pericardium_mesh\
        .voxelize_binary_mask(reference_volume = ref_volume)\
        .point_data['mask']\
        .reshape(cavity_all_mask.shape, order='F')
    
    return pericardium_mesh, pericardium_mask.astype(np.uint8)

# def plot_volume(data: np.ndarray, ref_nii: Nifti1Image, *, is_label: bool) -> None:
#     grid = pv.ImageData()
#     grid.origin = ref_nii.affine[:3, 3] if ref_nii.affine is not None else (0, 0, 0)
#     grid.spacing = ref_nii.header.get_zooms()[:3]
#     grid.dimensions = data.shape
#     grid.point_data["label"] = data.flatten(order="F")  # Fortran order for VTK
#     cmap = "viridis" if is_label else "bone"
#     opacity = "foreground" if is_label else [0, 0, 0, 0, 0.1, 0.3, 1]
#     grid.plot(show_scalar_bar=False, notebook=False, volume=True, cmap=cmap, opacity=opacity, show_bounds=True, shade=False)

# def plot_surface(data: np.ndarray) -> None:
#     assert data.dtype == np.uint8
#     labels = np.unique(data)
#     pl = pv.Plotter(notebook=False)
#     for label in labels:
#         if label == 0:
#             continue
#         mask = (data == label).astype(np.uint8)
#         mesh = pv.wrap(mask).contour([1], method="flying_edges").triangulate().smooth_taubin(n_iter=50).decimate_pro(
#             reduction=0.95,          # 减少 三角面片
#             preserve_topology=True, # 防止破洞
#             feature_angle=30.0
#         ).clean()
#         pl.add_mesh(mesh, name=f"Label {label}", show_edges=True, color=pv.Color(np.random.rand(3)))
#     pl.show()

In [96]:
pericardium_mesh, pericardium_mask = generate_pericardium(cavity_data, coronary_data)
pericardium_mesh.extract_all_edges().plot(notebook=False)

以下是函数实现的详细过程，包括更好的可视化效果

In [84]:
padding_voxel = 5.0
# 1. 将 numpy 转换为 PyVista 能够理解的 UniformGrid
# 假设 spacing 是 (1, 1, 1)，如有实际值请传入
cavity_all_mask = np.isin(cavity_data, list(CavityLabel)).astype(np.uint8)
cavity_add_coronary_mask = np.logical_or(cavity_all_mask, coronary_data > 0).astype(np.uint8)

# 2. 提取等值面 (Marching Cubes)
cavity_all_mesh: pv.PolyData = pv.wrap(cavity_add_coronary_mask)\
    .contour([1], method="flying_edges")\
    .triangulate()\
    .smooth_taubin()\
    .decimate_pro(
        reduction=0.95,          # 减少 三角面片
        preserve_topology=True, # 防止破洞
        # feature_angle=30.0
    )\
    .clean()

points = cavity_all_mesh.points
normals = cavity_all_mesh.point_normals
offset_points = points + normals * padding_voxel

pericardium_mesh = pv.PolyData(offset_points)\
    .delaunay_3d()\
    .extract_surface(algorithm=None)



In [ ]:
pl = pv.Plotter(notebook=False)
for label in CavityLabel:
    mask = (cavity_data == label).astype(np.uint8)
    mesh = pv.wrap(mask).contour([1], method="flying_edges").triangulate().smooth_taubin(n_iter=50).decimate_pro(
        reduction=0.95,          # 减少 三角面片
        preserve_topology=True, # 防止破洞
        feature_angle=30.0
    ).clean()
    pl.add_mesh(mesh, name=f"Label {label}", color=pv.Color(np.random.rand(3)), opacity=0.8)

coronary_mesh = pv.wrap(coronary_data).contour([1], method="flying_edges").triangulate().smooth_taubin(n_iter=50)
pl.add_mesh(coronary_mesh, name="Coronary", color="red", opacity=0.8)

pl.add_mesh(pericardium_mesh.extract_all_edges(), name="Pericardium", color="red")  #type: ignore
pl.show()

In [86]:
# 转回 Numpy Mask
ref_volume = pv.ImageData(dimensions=cavity_all_mask.shape)
pericardium_mask = pericardium_mesh\
    .voxelize_binary_mask(reference_volume = ref_volume)\
    .point_data['mask']\
    .reshape(cavity_all_mask.shape, order='F')

nib.save(nib.Nifti1Image(pericardium_mask.astype(np.uint8), cavity_img.affine), temp_output / "pericardium_mask.nii.gz")